In [1]:
import cobra

import pandas as pd

from Bio.Seq import Seq
from Bio import SeqIO
from Bio.Alphabet import generic_dna

import multiprocessing
from multiprocessing import Pool
from tqdm import tqdm

import scipy.stats as st
from bs4 import BeautifulSoup
import urllib
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import warnings

import requests, sys, json, re
sys.path.insert(1, '../scripts/') # comment out in python script
from load_environmental_variables import *
human_model = cobra.io.load_json_model(local_data_path + 'processed/corrected_model.json')

ModuleNotFoundError: No module named 'load_environmental_variables'

In [37]:
# MANE SELECTED transcripts and protein sequences
ids = pd.read_csv(local_data_path + 'raw/MANE.GRCh38.v0.9.summary.txt', sep = '\t')

psim_me = ids.loc[:, ['Ensembl_Gene', 'HGNC_ID', '#NCBI_GeneID', 'Ensembl_nuc', 'Ensembl_prot', 'symbol', 'name', 'chr_strand']]
psim_me.columns = ['ENSG_ID', 'HGNC_ID', 'NCBI_ID', 'ENST_ID', 'ENSP_ID', 'GENE_SYMBOL', 'GENE_NAME', 'CHR_STRAND']

polyA = pd.read_csv(local_data_path + 'processed/polyA_length.csv', index_col = 0)
psim_me['POLYA_LENGTH'] = psim_me.GENE_SYMBOL.map(dict(zip(polyA.index.tolist(), polyA.MEAN.tolist())))

sequence mapping

In [38]:
# add protein sequences
protein = list(SeqIO.parse(local_data_path + 'raw/MANE.GRCh38.v0.9.select_ensembl_protein.faa', "fasta"))
p_map = dict()
for p in protein:
    p_map[p.id] = str(p.seq)
psim_me['PROTEIN_SEQ'] = psim_me.ENSP_ID.map(p_map)

# add mrna sequence
mrna = list(SeqIO.parse(local_data_path + 'raw/MANE.GRCh38.v0.9.select_ensembl_rna.fna', "fasta"))
m_map = dict()
for m in mrna:
    m_map[m.id] = str(m.seq)
psim_me['MRNA_SEQ'] = psim_me.ENST_ID.map(m_map)

premrna more complicated because FTP doesn't have ENSG to full gene sequence

In [40]:
# # # add premrna sequence - no FTP file for this, parallelize REST API instead
# def get_premrna_seq(ensg_id, counter):
#     print(counter)
#     try:
#         hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?' 
#         return requests.get(hyperlink, headers={ "Content-Type" : "text/plain"}).text # introns and UTRs
#     except:
#         return float('nan')

# # pool = Pool(processes = multiprocessing.cpu_count())
# # premrna = pool.starmap(get_premrna_seq, zip(psim_me['ENSG_ID'].apply(lambda x: x.split('.')[0]).tolist(), list(range(psim_me.shape[0]))))
# # pool.close()

# # with open(local_data_path + 'interim/premrna_sequences.txt', 'w') as f:
# #     for seq in premrna:
# #         if type(seq) != str:
# #             seq = str(seq)
# #         f.write(seq + '\n')

# premrna = open(local_data_path + 'interim/premrna_sequences.txt', 'r').read().splitlines()
###NOT ALL DOWNLOADED, SO RERUNNING ON THOSE THAT DIDNT DOWNLOAD
# fail = 'You have exceeded the limit of 15 requests per second; please reduce your concurrent connections'
# fail_index = [i for i in range(len(premrna)) if premrna[i] == fail]
# pool = Pool(processes = 4)
# corrected = pool.starmap(get_premrna_seq, zip(psim_me.loc[fail_index, 'ENSG_ID'].apply(lambda x: x.split('.')[0]).tolist(), list(range(psim_me.shape[0]))))
# pool.close()
# for idx, seq in dict(zip(fail_index, corrected)).items():
#     premrna[idx] = seq

# with open(local_data_path + 'interim/premrna_sequences_v2.txt', 'w') as f:
#     for seq in premrna:
#         if type(seq) != str:
#             seq = str(seq)
#         f.write(seq + '\n')
premrna2 = open(local_data_path + 'interim/premrna_sequences_v2.txt', 'r').read().splitlines()
psim_me['PREMRNA_SEQ'] = premrna2

psim_me.loc[psim_me[psim_me.PREMRNA_SEQ == 'nan'].index, 'PREMRNA_SEQ'] = float('nan')

for i in psim_me.index:
    if type(psim_me.loc[i, 'PREMRNA_SEQ']) == str:
        if len(psim_me.loc[i,'MRNA_SEQ']) > len(psim_me.loc[i,'PREMRNA_SEQ']):
            psim_me.loc[i,'PREMRNA_SEQ'] = float('nan')
            

In [41]:
# formatting
def transcribe(x):
    try: 
        return str(Seq(x).transcribe())
    except:
        return float('nan')

psim_me['MRNA_SEQ'] = psim_me['MRNA_SEQ'].apply(lambda x: transcribe(x))
psim_me['PREMRNA_SEQ'] = psim_me['PREMRNA_SEQ'].apply(lambda x: transcribe(x))

In [45]:
uniprot_map = pd.read_csv(local_data_path + 'raw/gencode.v34.metadata.SwissProt', sep = '\t', header = None)
psim_me['UNIPROT_ID'] = psim_me['ENST_ID'].map(dict(zip(uniprot_map[0].tolist(), uniprot_map[1].tolist())))

psim_human = pd.read_csv(root_path + 'MammalianSecretoryRecon/JUPYTER_NOTEBOOKS/RECON2s_python/PSIM_HUMAN.tab', 
                         sep = '\t')
cols = ['SP', 'DSB', 'GPI', 'NG', 'OG', 'TMD', 'Location']
for col in cols:
    psim_me[col] = psim_me['UNIPROT_ID'].map(dict(zip(psim_human.Entry.tolist(), psim_human[col].tolist())))
    
# for i in psim_me.Location.dropna().index:
#     a = psim_me.loc[i, 'Location']
#     psim_me.loc[i, 'Location'] = a.split('[')[1].split(']')[0]

# To Do

For machinery that don't have a MANE transcript, develop an alternative method to get the PSIM information. For now, I am taking the first result from the ENSEMBL rest api to build some of the necessary reactions but later will change this to the general method. 

In [ ]:
def get_premrna_seq(ensg_id):
    try:
        hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?type=genomic' 
        return requests.get(hyperlink, headers={ "Content-Type" : "text/plain"}).text 
    except:
        return float('nan')

def get_mrna_seq(ensg_id):
    try:
        hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?type=cdna;multiple_sequences=1' 
        return requests.get(hyperlink, headers={ "Content-Type" : "text/plain"}).text 
    except:
        return float('nan')

def get_protein_seq(ensg_id):
    try:
        hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?type=protein;multiple_sequences=1' 
        return requests.get(hyperlink, headers={ "Content-Type" : "text/plain"}).text 
    except:
        return float('nan')
    
def get_all_seq(ensg_id):
    return get_premrna_seq(ensg_id), get_mrna_seq(ensg_id).splitlines()[0], get_protein_seq(ensg_id).splitlines()[0]

In [8]:
# # RPS27a
# cols = ['ENSG_ID', 'HGNC_ID', 'GENE_SYMBOL', 'PROTEIN_SEQ', 'MRNA_SEQ', 'PREMRNA_SEQ', 'POLYA_LENGTH']
# ensg_id, hgnc_id, gene_name = 'ENSG00000143947', 'HGNC:10417', 'RPS27A'
# premrna, mrna, protein = get_all_seq(ensg_id)
# polyA_L = polyA.loc[gene_name,'MEAN']
# psim_me.loc[psim_me.shape[0],:] = [float('nan')]*psim_me.shape[1]
# psim_me.loc[psim_me.shape[0]-1,cols] = [ensg_id, hgnc_id, gene_name, protein, transcribe(mrna), 
#                                         transcribe(premrna),polyA_L]

# # RPLs
# cols = ['ENSG_ID', 'HGNC_ID', 'PROTEIN_SEQ', 'MRNA_SEQ', 'PREMRNA_SEQ', 'POLYA_LENGTH']
# rl_missing_hgnc = ['HGNC:10307', 'HGNC:10313', 'HGNC:10340', 'HGNC:10362', 'HGNC:10368']
# rl_missing_ensg = ['ENSG00000265681', 'ENSG00000122026', 'ENSG00000109475', 'ENSG00000089009', 'ENST00000262584']
# for i in range(len(rl_missing_hgnc)):
#     ensg_id, hgnc_id = rl_missing_ensg[i], rl_missing_hgnc[i]
#     premrna, mrna, protein = get_all_seq(ensg_id)
#     try:
#         polyA_L = polyA.loc[gene_name,'MEAN']
#     except:
#         polyA_l = float('nan')
    
#     psim_me.loc[psim_me.shape[0],:] = [float('nan')]*psim_me.shape[1]
#     psim_me.loc[psim_me.shape[0]-1,cols] = [ensg_id, hgnc_id, protein, transcribe(mrna), 
#                                             transcribe(premrna),polyA_L]

# You are here
must fill out missing machinery sequences, 

-right now just take first transcript but must improve this

In [71]:
metabolic_enzymes = set([g.id for g in human_model.genes])
missing_genes = sorted(metabolic_enzymes.difference(psim_me.HGNC_ID.tolist()))

ehm = pd.read_csv(local_data_path + 'raw/identifiers.txt', sep = '\t')
ehm = ehm.loc[ehm['NCBI gene ID'].dropna().index,:]
ehm['NCBI gene ID'] = ehm['NCBI gene ID'].astype('int64').astype(str)
ehm = ehm[ehm['HGNC ID'].isin(missing_genes)]

if len(set(missing_genes).difference(ehm['HGNC ID'].unique().tolist() + ['ribosome'])) > 0:
    raise ValueError('Not all missing genes are maping, make sure you add them')
    
ens_hgnc = dict(zip(ehm['HGNC ID'].tolist(), ehm['Ensembl gene ID'].tolist()))
name_hgnc = dict(zip(ehm['HGNC ID'].tolist(), ehm['Approved symbol'].tolist()))

In [10]:
still_missing_genes = list()
for gene in tqdm(missing_genes):
    ensg_id = ens_hgnc[gene]
    gene_name = name_hgnc[gene]
    try:
        premrna_seq, mrna_seq, protein_seq = get_all_seq(ensg_id)
        premrna_seq, mrna_seq = transcribe(premrna_seq), transcribe(mrna_seq)
        
        
        if type(premrna_seq) == str and type(mrna_seq) == str and type(protein_seq) == str: 
            counter = psim_me.shape[0]
            
            if gene_name in polyA.index:
                polya_l = polyA.loc[gene_name, 'MEAN']
            else:
                polya_l = float('nan')
                
            row = [float('nan'), gene] + [float('nan')]*6 + [polya_l] + [protein_seq, mrna_seq, premrna_seq]
            row += [float('nan')]*8

            psim_me.loc[counter,:] = row
        else:
            still_missing_genes.append(gene)
    except:
        still_missing_genes.append(gene)

100%|██████████| 202/202 [46:52<00:00, 13.92s/it] 


In [11]:
if len(still_missing_genes) > 0:
    raise ValueError('Some genes still have not been added')

In [12]:
def convert_to_list(x):
    if not pd.isna(x):
        x = list(x)
    else:
        x = x
    return x


In [97]:
ps = 'MSGYDRMLRTLGGNLMEFIENLDALHSYLALSYQEMNAPSFRVERGADGKMFLHYYSDRSGLCHIVPGIIEAVAKDFFDIDVIMDILDMNEEVERTGKKEHVVFLIVQKAHRKMRKTKPKRLQDSQGMERDQEALQAAFLKMKEKYLNVSACPVKKSHWDVVRSIVMFGKGHLMNTFEPIYPERLWIEEKTFCNAFPFHIVFDESLQVKQARVNIQKYVPGLQTQNIQLDEYFSIIHPQVTFNIFSIRRFINSQFVLKTRREMMPVAWQSRTTLKLQGQMIWMESMWCMVYLCSPKLRSLQELEELNMHLSDIAPNDTTRDLILLNQQRLAEIELSNQLERKKEELQVLSKHLAIEKKKTETLLYAMLPKHVANQLREGKKVAAGEFKSCTILFSDVVTFTNICTACEPIQIVNVLNSMYSKFDRLTSVHAVYKVETIGDAYMVVGGVPVPIGNHAQRVANFALGMRISAKEVTNPVTGEPIQLRVGIHTGPVLADVVGDKMPRYCLFGDTVNTASRMESHGLPNKVHLSPTAYRALKNQGFKIIERGEIEVKGKGRMTTYFLIQNLNATEDEIMGRSKTPVDHKGSTQKASLPTTKLQGSVQPSCPEHSSLASWLL'
psim_me.loc[psim_me[psim_me.HGNC_ID == 'HGNC:4686'].index, 'PROTEIN_SEQ'] = ps

In [13]:
psim_me = psim_me[['HGNC_ID', 'POLYA_LENGTH', 'PROTEIN_SEQ', 'MRNA_SEQ', 'PREMRNA_SEQ', 
                  'SP', 'DSB', 'GPI', 'NG', 'OG', 'TMD', 'Location']]
psim_me['SP'] = psim_me['SP'].map({1: True, 0: False})
psim_me.columns = ['HGNC_ID', 'POLYA_LENGTH', 'PROTEIN_SEQ', 'MRNA_SEQ', 'PREMRNA_SEQ', 
                  'SP', 'DSB', 'GPI', 'NG', 'OG', 'TMD', 'LOCATION']
psim_me['N_INTRONS'] = float('nan')
psim_me.LOCATION = psim_me.LOCATION.apply(lambda x: convert_to_list(x))
psim_me.to_csv(root_path + 'psim_recon2_2.csv')

In [16]:
# # need to do ribosomes as well before the build_psim_expression script, since that relies on build_ribosome..
# missing_genes = ['HGNC:10417', 'HGNC:10307', 'HGNC:10313', 'HGNC:10340', 'HGNC:10362', 'HGNC:10368']
# ehm = pd.read_csv(local_data_path + 'raw/identifiers.txt', sep = '\t')
# ehm = ehm.loc[ehm['NCBI gene ID'].dropna().index,:]
# ehm['NCBI gene ID'] = ehm['NCBI gene ID'].astype('int64').astype(str)
# ehm = ehm[ehm['HGNC ID'].isin(missing_genes)]

# if len(set(missing_genes).difference(ehm['HGNC ID'].unique().tolist() + ['ribosome'])) > 0:
#     raise ValueError('Not all missing genes are maping, make sure you add them')
    
# ens_hgnc = dict(zip(ehm['HGNC ID'].tolist(), ehm['Ensembl gene ID'].tolist()))
# name_hgnc = dict(zip(ehm['HGNC ID'].tolist(), ehm['Approved symbol'].tolist()))

# psim_2 = psim_me.copy()

# still_missing_genes = list()
# for gene in tqdm(['HGNC:10417', 'HGNC:10307', 'HGNC:10313', 'HGNC:10340', 'HGNC:10362', 'HGNC:10368']):
#     ensg_id = ens_hgnc[gene]
#     gene_name = name_hgnc[gene]
#     try:
#         premrna_seq, mrna_seq, protein_seq = get_all_seq(ensg_id)
#         premrna_seq, mrna_seq = transcribe(premrna_seq), transcribe(mrna_seq)
#         if type(premrna_seq) == str and type(mrna_seq) == str and type(protein_seq) == str: 
#             counter = psim_2.shape[0]
            
#             if gene_name in polyA.index:
#                 polya_l = polyA.loc[gene_name, 'MEAN']
#             else:
#                 polya_l = float('nan')
                
#             row = [gene, polya_l, protein_seq, mrna_seq, premrna_seq]
#             row += [float('nan')]*8

#             psim_2.loc[counter,:] = row
#         else:
#             still_missing_genes.append(gene)
#     except:
#         still_missing_genes.append(gene)
# psim_2.to_csv(root_path+'TRASH.csv')

100%|██████████| 6/6 [02:48<00:00, 28.10s/it]


# add n_introns